# YOLO Multi-Model Mask Overlay (Colab)

Use two YOLO models on the same video and merge their masks.

1. Upload model **A** and model **B** (`.pt` each). Available classes are printed for each.
2. Upload a video.
3. Pick which classes to keep from each model and (optionally) alias multiple raw class names to a single display label that shares one color.
4. Render an annotated MP4. Names persist for a few frames after a detection drops to stop the flicker.
5. Download.

In [ ]:
!pip install -q ultralytics opencv-python

In [ ]:
import tempfile
from pathlib import Path

import cv2
import numpy as np
from google.colab import files
from ultralytics import YOLO

WORK_DIR = Path(tempfile.mkdtemp(prefix="yolo_overlay_"))
SLOTS = {"A": None, "B": None}  # each: {name, model, names, keep_ids}
VIDEO_PATH = None
print(f"Working directory: {WORK_DIR}")

## 1. Upload model A

In [ ]:
uploaded = files.upload()
name = next(iter(uploaded))
p = WORK_DIR / f"A_{name}"
p.write_bytes(uploaded[name])
m = YOLO(str(p))
SLOTS["A"] = {"name": name, "model": m, "names": m.names, "keep_ids": []}
print(f"\nModel A: {name}  (task={getattr(m, 'task', '?')})")
print("Classes:")
for i, n in m.names.items():
    print(f"  {i:>3}: {n}")

## 2. Upload model B

In [ ]:
uploaded = files.upload()
name = next(iter(uploaded))
p = WORK_DIR / f"B_{name}"
p.write_bytes(uploaded[name])
m = YOLO(str(p))
SLOTS["B"] = {"name": name, "model": m, "names": m.names, "keep_ids": []}
print(f"\nModel B: {name}  (task={getattr(m, 'task', '?')})")
print("Classes:")
for i, n in m.names.items():
    print(f"  {i:>3}: {n}")

## 3. Upload video

In [ ]:
uploaded = files.upload()
name = next(iter(uploaded))
vp = WORK_DIR / name
vp.write_bytes(uploaded[name])
VIDEO_PATH = str(vp)

cap = cv2.VideoCapture(VIDEO_PATH)
n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f"Loaded {name}: {w}x{h}, {fps:.2f} fps, {n} frames")

## 4. Pick classes per model and aliases

- `KEEP_A` / `KEEP_B`: comma-separated class names or ids from each model (case-insensitive).
- `ALIASES`: optional. Maps any raw class name (from either model) to a display label. Classes that share a display label share one color and merge into one mask layer. The display label is what gets drawn on the video.

In [ ]:
KEEP_A = "rough trail, rocky trail"
KEEP_B = "rock"

ALIASES = {
    "rough trail": "trail",
    "rocky trail": "trail",
}

def _resolve(keep_str, names):
    name_to_id = {n.lower(): i for i, n in names.items()}
    out = []
    for tok in keep_str.split(","):
        t = tok.strip().lower()
        if not t:
            continue
        if t.isdigit() and int(t) in names:
            out.append(int(t))
        elif t in name_to_id:
            out.append(name_to_id[t])
        else:
            raise ValueError(f"Unknown class in this model: {tok!r}")
    return sorted(set(out))

if SLOTS["A"] is not None:
    SLOTS["A"]["keep_ids"] = _resolve(KEEP_A, SLOTS["A"]["names"])
if SLOTS["B"] is not None:
    SLOTS["B"]["keep_ids"] = _resolve(KEEP_B, SLOTS["B"]["names"])

_aliases_lc = {k.lower(): v for k, v in ALIASES.items()}
def display_label(raw_name):
    return _aliases_lc.get(raw_name.lower(), raw_name)

for slot_name, slot in SLOTS.items():
    if slot is None:
        continue
    print(f"Model {slot_name} ({slot['name']}) keeping:")
    for i in slot["keep_ids"]:
        raw = slot["names"][i]
        lbl = display_label(raw)
        suffix = f"  ->  {lbl}" if lbl != raw else ""
        print(f"  {i:>3}: {raw}{suffix}")
    print()

## 5. Render annotated video

Tweakables:
- `CONF`: detection confidence threshold (per model).
- `ALPHA`: mask overlay opacity.
- `STICKY_FRAMES`: how long a label keeps showing after the detection disappears.
- `EMA_ALPHA`: how fast the label position eases to a new centroid (0 = frozen, 1 = no easing).
- `MATCH_DIST`: max pixel distance to consider a current blob the same as a tracked label position.

In [ ]:
CONF = 0.25
ALPHA = 0.5
STICKY_FRAMES = 12
EMA_ALPHA = 0.25
MATCH_DIST = 120
MIN_BLOB_AREA = 200

def color_for(label):
    seed = sum(ord(ch) for ch in label) * 9973 + 7
    rng = np.random.default_rng(seed)
    return tuple(int(c) for c in rng.integers(64, 255, size=3))

active_slots = [s for s in SLOTS.values() if s is not None]
if not active_slots:
    raise RuntimeError("Upload at least one model.")
if VIDEO_PATH is None:
    raise RuntimeError("Upload a video.")
if not any(s["keep_ids"] for s in active_slots):
    raise RuntimeError("Pick at least one class via KEEP_A/KEEP_B.")

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

out_path = WORK_DIR / "output_overlay.mp4"
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(out_path), fourcc, fps, (W, H))

label_tracks = {}  # display_label -> list of {"centroid": (x,y), "missed": int}

def update_tracks(label, current_centroids):
    tracks = label_tracks.setdefault(label, [])
    pairs = []
    for ci, c in enumerate(current_centroids):
        for ti, t in enumerate(tracks):
            d = float(np.hypot(c[0] - t["centroid"][0], c[1] - t["centroid"][1]))
            pairs.append((d, ci, ti))
    pairs.sort()
    used_c, used_t = set(), set()
    for d, ci, ti in pairs:
        if d > MATCH_DIST:
            break
        if ci in used_c or ti in used_t:
            continue
        cx, cy = current_centroids[ci]
        tx, ty = tracks[ti]["centroid"]
        tracks[ti]["centroid"] = (tx + EMA_ALPHA * (cx - tx), ty + EMA_ALPHA * (cy - ty))
        tracks[ti]["missed"] = 0
        used_c.add(ci); used_t.add(ti)
    for ci, c in enumerate(current_centroids):
        if ci not in used_c:
            tracks.append({"centroid": (float(c[0]), float(c[1])), "missed": 0})
    for ti, t in enumerate(tracks):
        if ti not in used_t:
            t["missed"] += 1
    label_tracks[label] = [t for t in tracks if t["missed"] <= STICKY_FRAMES]

frame_idx = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break

    label_masks = {}  # display_label -> binary mask (H, W)

    for slot in active_slots:
        if not slot["keep_ids"]:
            continue
        is_seg = getattr(slot["model"], "task", None) == "segment"
        result = slot["model"].predict(frame, conf=CONF, classes=slot["keep_ids"], verbose=False)[0]
        boxes = result.boxes
        masks = result.masks if is_seg else None
        if boxes is None or len(boxes) == 0:
            continue
        cls = boxes.cls.cpu().numpy().astype(int)
        xyxy = boxes.xyxy.cpu().numpy().astype(int)
        mask_data = masks.data.cpu().numpy() if masks is not None else None

        for i in range(len(cls)):
            c = int(cls[i])
            if c not in slot["keep_ids"]:
                continue
            lbl = display_label(slot["names"][c])

            if mask_data is not None:
                m = mask_data[i]
                if m.shape != (H, W):
                    m = cv2.resize(m, (W, H), interpolation=cv2.INTER_LINEAR)
                m = (m >= 0.5).astype(np.uint8)
            else:
                x1, y1, x2, y2 = xyxy[i]
                m = np.zeros((H, W), dtype=np.uint8)
                m[max(0, y1):max(0, y2), max(0, x1):max(0, x2)] = 1

            if lbl in label_masks:
                label_masks[lbl] = np.maximum(label_masks[lbl], m)
            else:
                label_masks[lbl] = m

    if label_masks:
        overlay = frame.astype(np.float32)
        any_mask = np.zeros((H, W), dtype=np.float32)
        color_layer = np.zeros_like(overlay)
        weight = np.zeros((H, W), dtype=np.float32)
        for lbl, m in label_masks.items():
            color = np.array(color_for(lbl), dtype=np.float32)
            mf = m.astype(np.float32)
            color_layer += mf[..., None] * color
            weight += mf
            any_mask = np.maximum(any_mask, mf)
        w_safe = np.clip(weight, 1.0, None)[..., None]
        color_layer = color_layer / w_safe
        a = (ALPHA * any_mask)[..., None]
        frame = np.clip(overlay * (1 - a) + color_layer * a, 0, 255).astype(np.uint8)

    # update label tracks (sticky labels): every label that COULD appear gets a tick
    seen_labels = set(label_masks.keys())
    current_centroids_by_label = {}
    for lbl, m in label_masks.items():
        num, _, stats, centroids = cv2.connectedComponentsWithStats(m, connectivity=8)
        cs = []
        for k in range(1, num):
            if stats[k, cv2.CC_STAT_AREA] < MIN_BLOB_AREA:
                continue
            cs.append((float(centroids[k][0]), float(centroids[k][1])))
        current_centroids_by_label[lbl] = cs
    for lbl in set(list(label_tracks.keys()) + list(current_centroids_by_label.keys())):
        update_tracks(lbl, current_centroids_by_label.get(lbl, []))

    # draw labels (current + sticky)
    for lbl, tracks in label_tracks.items():
        color = color_for(lbl)
        for t in tracks:
            cx, cy = int(t["centroid"][0]), int(t["centroid"][1])
            font = cv2.FONT_HERSHEY_SIMPLEX
            scale = 0.6
            thickness = 2
            (tw, th), baseline = cv2.getTextSize(lbl, font, scale, thickness)
            x = max(0, min(W - tw - 4, cx - tw // 2))
            y = max(th + 4, min(H - 4, cy + th // 2))
            cv2.rectangle(frame, (x - 2, y - th - 4), (x + tw + 2, y + baseline), (0, 0, 0), -1)
            cv2.putText(frame, lbl, (x, y), font, scale, color, thickness, cv2.LINE_AA)

    writer.write(frame)
    frame_idx += 1
    if frame_idx % 20 == 0 or frame_idx == TOTAL:
        print(f"frame {frame_idx}/{TOTAL}")

cap.release()
writer.release()
print(f"\nWrote {out_path}")

## 6. Download

In [ ]:
files.download(str(out_path))